# Video Sync Per-Interval Orchestrator

Replaces the GNU Parallel batch flow (`run_video_sync_parallel_slurm.sh`) with a per-interval SLURM orchestration pattern.

For a chosen patient and camera serial, this notebook:
1. Discovers all interval directories under `vad_new/{patient}/vad_data`
2. Checks which intervals already have video sync output (from old or new pipeline)
3. Submits one SLURM job per pending interval (chipmunk partition, CPU-only)
4. Lets you inspect status and errors

Worker script:  
[notebooks/REBIRTH_2!/video_processing/video_sync_worker.py](/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/video_sync_worker.py)

In [1]:
from pathlib import Path
import subprocess
import pandas as pd

In [11]:
# --------------------
# User-facing settings
# --------------------
PROJ_DIR     = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT     = PROJ_DIR / 'vad_new'

PATIENT      = 'YFL'
CAM_SERIAL   = '23512908'
KEYWORDS     = 'neural'
LOG_LEVEL    = 'DEBUG'
FORCE_RESUBMIT = True   # set True to rerun even intervals with existing output
MAX_INTERVALS  = None    # e.g. 3 for a quick test run

VIDEO_DIR    = Path('/mnt/datalake/data/emu/YFLDatafile/VIDEO')
REPO_ROOT    = Path('/scratch/tahaismail424/video-sync-nbu')
WORKER_PATH  = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/video_sync_worker.py')
PYTHON_BIN   = '/scratch/tahaismail424/miniforge3/envs/video_sync_db/bin/python3'
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate video_sync_db'

# SLURM settings (chipmunk = CPU-only partition)
PARTITION  = 'Universal'
CPUS       = 8
MEM        = '32G'
TIME       = '24:00:00'
QOS        = 'big_batch_tier'   # e.g. 'default_tier', 'big_batch_tier'

RUN_NAME   = f'video_sync_{PATIENT}'
VAD_DATA_DIR = VAD_ROOT / PATIENT / 'vad_data'
LOGS_DIR   = VAD_ROOT / PATIENT / 'video_sync_slurm_logs'
SCRIPTS_DIR = VAD_ROOT / PATIENT / 'video_sync_slurm_scripts'
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('patient:     ', PATIENT)
print('cam_serial:  ', CAM_SERIAL)
print('vad_data:    ', VAD_DATA_DIR)
print('video_dir:   ', VIDEO_DIR)
print('repo_root:   ', REPO_ROOT)
print('worker:      ', WORKER_PATH)
print('logs_dir:    ', LOGS_DIR)
print('scripts_dir: ', SCRIPTS_DIR)

patient:      YFL
cam_serial:   23512908
vad_data:     /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFL/vad_data
video_dir:    /mnt/datalake/data/emu/YFLDatafile/VIDEO
repo_root:    /scratch/tahaismail424/video-sync-nbu
worker:       /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/video_sync_worker.py
logs_dir:     /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFL/video_sync_slurm_logs
scripts_dir:  /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFL/video_sync_slurm_scripts


## 1 — Scan intervals

In [12]:
def derive_status_row(interval_dir: Path) -> dict:
    interval_id = interval_dir.name
    pipeline_out_dir = interval_dir / 'video' / interval_id
    success_path = pipeline_out_dir / '_SUCCESS'
    error_path   = pipeline_out_dir / '_ERROR'
    has_output   = pipeline_out_dir.exists()
    has_success  = success_path.exists()
    has_error    = error_path.exists()
    # Synced mp4 produced by the pipeline
    synced_mp4 = (
        pipeline_out_dir / 'neural' / CAM_SERIAL / 'synced_video' / f'neural_{CAM_SERIAL}.mp4'
    )
    has_mp4 = synced_mp4.exists()

    if FORCE_RESUBMIT:
        ready = True
    else:
        ready = not has_output

    return {
        'interval_id':       interval_id,
        'interval_dir':      interval_dir,
        'pipeline_out_dir':  pipeline_out_dir,
        'success_path':      success_path,
        'error_path':        error_path,
        'has_output':        has_output,
        'has_success':       has_success,
        'has_error':         has_error,
        'has_mp4':           has_mp4,
        'ready_for_submit':  ready,
    }


interval_dirs = sorted(VAD_DATA_DIR.glob('202*'))
rows = [derive_status_row(d) for d in interval_dirs if d.is_dir()]
status_df = pd.DataFrame(rows)

if MAX_INTERVALS is not None:
    status_df = status_df.head(MAX_INTERVALS).copy()

print(f'intervals found:      {len(status_df)}')
print(f'has output (any):     {status_df["has_output"].sum()}')
print(f'has synced mp4:       {status_df["has_mp4"].sum()}')
print(f'has _SUCCESS:         {status_df["has_success"].sum()}')
print(f'has _ERROR:           {status_df["has_error"].sum()}')
print(f'ready_for_submit:     {status_df["ready_for_submit"].sum()}')

status_df[['interval_id', 'has_output', 'has_mp4', 'has_success', 'has_error', 'ready_for_submit']]

intervals found:      765
has output (any):     765
has synced mp4:       472
has _SUCCESS:         471
has _ERROR:           293
ready_for_submit:     765


,interval_id,has_output,has_mp4,has_success,has_error,ready_for_submit
0,20250224-201223_0000,True,False,False,True,True
1,20250224-201223_0001,True,False,False,True,True
2,20250224-201223_0002,True,False,False,True,True
3,20250224-201223_0003,True,False,False,True,True
4,20250224-201223_0004,True,False,False,True,True
...,...,...,...,...,...,...
760,20250306-095216_0004,True,True,True,False,True
761,20250306-095216_0005,True,True,True,False,True
762,20250306-095216_0006,True,True,True,False,True
763,20250306-095216_0007,True,True,True,False,True


## 2 — Submit SLURM jobs

In [8]:
submitted = []
failed    = []

to_submit = status_df[(status_df['ready_for_submit'])]
print(f'Submitting {len(to_submit)} jobs...')

for _, row in to_submit.iterrows():
    interval_id = row['interval_id']

    reset_line = f'rm -f {row["success_path"]} {row["error_path"]}' if FORCE_RESUBMIT else 'true'

    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name=vsync_{interval_id}',
        f'#SBATCH --partition={PARTITION}',
        f'#SBATCH --qos={QOS}',
        f'#SBATCH --cpus-per-task={CPUS}',
        f'#SBATCH --mem={MEM}',
        f'#SBATCH --time={TIME}',
        f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --output={LOGS_DIR}/{interval_id}_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{interval_id}_%j.err',
        '',
        'set -eo pipefail',
        '',
        CONDA_ACTIVATE,
        '',
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT: {PATIENT}"',
        f'echo "INTERVAL_ID: {interval_id}"',
        '',
        reset_line,
        '',
        f'{PYTHON_BIN} {WORKER_PATH} \\',
        f'    {row["interval_dir"]} \\',
        f'    {REPO_ROOT} \\',
        f'    {VIDEO_DIR} \\',
        f'    {CAM_SERIAL} \\',
        f'    --keywords {KEYWORDS} \\',
        f'    --log-level {LOG_LEVEL}',
        '',
        'echo "END: $(date)"',
    ]
    sbatch_text = '\n'.join(lines) + '\n'

    sbatch_path = SCRIPTS_DIR / f'{interval_id}.sbatch'
    sbatch_path.write_text(sbatch_text)

    try:
        res = subprocess.run(
            ['sbatch', str(sbatch_path)],
            capture_output=True, text=True, check=True
        )
        submitted.append((interval_id, res.stdout.strip()))
        print(f'submitted: {interval_id} -> {res.stdout.strip()}')
    except subprocess.CalledProcessError as exc:
        failed.append((interval_id, exc.stderr.strip()))
        print(f'FAILED SUBMISSION: {interval_id}')
        print(exc.stderr)

print(f'\nsubmitted={len(submitted)}, failed={len(failed)}')

Submitting 765 jobs...
submitted: 20250224-201223_0000 -> Submitted batch job 252593
submitted: 20250224-201223_0001 -> Submitted batch job 252594
submitted: 20250224-201223_0002 -> Submitted batch job 252595
submitted: 20250224-201223_0003 -> Submitted batch job 252596
submitted: 20250224-201223_0004 -> Submitted batch job 252597
submitted: 20250224-201223_0005 -> Submitted batch job 252598
submitted: 20250224-201223_0006 -> Submitted batch job 252599
submitted: 20250224-201223_0007 -> Submitted batch job 252600
submitted: 20250224-201223_0008 -> Submitted batch job 252601
submitted: 20250224-201223_0009 -> Submitted batch job 252602
submitted: 20250224-201223_0010 -> Submitted batch job 252603
submitted: 20250224-201223_0011 -> Submitted batch job 252604
submitted: 20250224-201223_0012 -> Submitted batch job 252605
submitted: 20250224-201223_0013 -> Submitted batch job 252606
submitted: 20250224-201223_0014 -> Submitted batch job 252607
submitted: 20250224-201223_0015 -> Submitted ba

## 3 — Check job status

In [ ]:
# Re-scan after jobs have run.
interval_dirs = sorted(VAD_DATA_DIR.glob('202*'))
rows = [derive_status_row(d) for d in interval_dirs if d.is_dir()]
status_df = pd.DataFrame(rows)

print(f'intervals found:      {len(status_df)}')
print(f'has output (any):     {status_df["has_output"].sum()}')
print(f'has synced mp4:       {status_df["has_mp4"].sum()}')
print(f'has _SUCCESS:         {status_df["has_success"].sum()}')
print(f'has _ERROR:           {status_df["has_error"].sum()}')
print(f'still pending:        {status_df["ready_for_submit"].sum()}')

status_df[['interval_id', 'has_output', 'has_mp4', 'has_success', 'has_error', 'ready_for_submit']]

In [ ]:
# Inspect errors.
error_rows = status_df[status_df['has_error']]
print(f'intervals with _ERROR: {len(error_rows)}')
for _, row in error_rows.head(5).iterrows():
    print('=' * 100)
    print(row['interval_id'])
    print(row['error_path'].read_text()[:4000])